# Bonus 05 — PydanticAI typed agent boundaries

Many agent frameworks begin with orchestration: agents, teams, graphs, or handoffs. PydanticAI begins from a different question:

> What types, dependencies, validation rules, limits, and tests must surround one model-driven component?

You will build a data-pipeline incident agent that:

- receives typed application dependencies without automatically exposing them to the model;
- calls a typed lookup tool through `RunContext`;
- repairs an invalid tool argument after `ModelRetry`;
- returns a validated `IncidentDecision`, not a JSON string;
- enforces business invariants with an output validator;
- caps requests, tool calls, and tokens;
- exposes the complete message protocol;
- runs a deterministic no-network test through `FunctionModel`.

This optional lab uses its own locked environment. It does not change modules 00–17 or the core course environment.

## 1. Learn — type the boundaries around the loop

```mermaid
flowchart LR
    U["User: investigate nightly-orders"] --> A["Agent[IncidentDeps, IncidentDecision]"]
    D["Typed dependencies"] --> C["RunContext[IncidentDeps]"]
    C --> I["Selected dynamic instructions"]
    C --> T["lookup_pipeline_run tool"]
    A --> M["OpenAI model"]
    M -->|"alias tool call"| T
    T -->|"ModelRetry: use run-204"| M
    M -->|"canonical tool call"| T
    T -->|"evidence"| M
    M --> O["IncidentDecision schema"]
    O --> V{"Output validator"}
    V -->|"ModelRetry if unsafe"| M
    V -->|"valid typed object"| APP["Application"]
```

PydanticAI does not make the model type-safe. It makes the **application boundary** explicit and validates what crosses it. The model can still propose an invalid argument or decision; the framework turns that failure into a structured repair opportunity.

### The type-safety map

| Surface | What it gives you | What it does not guarantee |
|---|---|---|
| `deps_type` | A declared type for run-scoped services and data | That dependencies remain secret if instructions or tools expose them |
| `RunContext[Deps]` | Typed access to dependencies, usage, retry state, and run metadata | Authorization by itself |
| typed tool signature | JSON schema plus validated arguments | That the requested action is permitted |
| `output_type` | Parsed, validated Python output | Factual or policy correctness |
| output validator | Deterministic business checks and a repair path | That retries will eventually succeed |
| `UsageLimits` | Hard bounds on requests, tokens, tool calls, or cost | A complete organizational budget system |
| message objects | Inspectable request, response, tool, and retry protocol | Durable governed storage |
| `FunctionModel` | Deterministic execution of application logic without an LLM | A quality evaluation of the production model |

The framework is deliberately thin. It is useful when Python types and dependency injection are more important than a visual workflow or a multi-agent abstraction.

### When this style fits

PydanticAI is a strong fit when a team already uses Python type hints and Pydantic models, wants dependencies to be explicit, and needs model calls to sit inside ordinary testable application code.

It is not automatically the best choice for every agent:

- a fixed transformation may still be plain Python or a non-agent pipeline;
- a durable branching workflow may be clearer as an explicit graph;
- a hand-written loop may be better when every protocol decision must remain visible;
- a framework does not remove the need for authorization, persistence, evaluation, or operational ownership.

The core course supports the no-framework path. This bonus lab shows what a thin typed framework can add without erasing that mental model.

### Why this lab is isolated

The lab installs `pydantic-ai-slim[openai]`, not the full package with every provider, CLI, MCP, web UI, eval, and observability integration. The slim environment still resolves 56 installed packages and OpenAI 3.3.1, so it must not be merged into the core course environment.

From this directory, prepare it once with:

```bash
uv sync --locked
```

In VS Code, select `bonus/05_pydanticai_typed_agents/.venv/bin/python` as the notebook kernel. The notebook never installs packages.

## 2. Do — build one typed incident agent

Load the course model and prices from the repository `.env`. The setup prints versions and model names, never credentials.

In [ ]:
import os
from dataclasses import dataclass, field
from importlib.metadata import version
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "modules").exists()
)
load_dotenv(repo_root / ".env")

MODEL_DEFAULT = os.getenv("MODEL_DEFAULT")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing from the repository .env"
assert MODEL_DEFAULT, "MODEL_DEFAULT is missing from the repository .env"

PRICE_INPUT = float(os.getenv("PRICE_INPUT_PER_MILLION", "0"))
PRICE_OUTPUT = float(os.getenv("PRICE_OUTPUT_PER_MILLION", "0"))

print("pydantic-ai-slim:", version("pydantic-ai-slim"))
print("pydantic:", version("pydantic"))
print("openai:", version("openai"))
print("course model:", MODEL_DEFAULT)

### Pin the provider surface deliberately

PydanticAI's `openai:` model prefix uses OpenAI's Responses API by default. Its `openai-chat:` prefix or `OpenAIChatModel` class pins Chat Completions.

This lab uses `OpenAIChatModel` so the tool-call messages connect directly to the protocol already taught in the core course. `OpenAIChatModelSettings` is the framework's typed settings surface. We pass the required OpenAI reasoning setting and do not pass `temperature`.

In [ ]:
from pydantic_ai import (
    Agent,
    ModelResponse,
    ModelRetry,
    RunContext,
    ToolCallPart,
    UsageLimits,
    capture_run_messages,
    models,
)
from pydantic_ai.messages import RetryPromptPart, ToolReturnPart
from pydantic_ai.models.function import AgentInfo, FunctionModel
from pydantic_ai.models.openai import OpenAIChatModel, OpenAIChatModelSettings

openai_model = OpenAIChatModel(MODEL_DEFAULT)
openai_settings = OpenAIChatModelSettings(openai_reasoning_effort="none")

print(type(openai_model).__name__)
print(openai_settings)

### Dependencies are application objects, not prompt text

`IncidentDeps` holds an environment name, operational records, aliases, allowed owners, and local audit lists. Passing an instance to `agent.run(..., deps=...)` does not automatically serialize it into the prompt.

Later, a dynamic instruction deliberately exposes the environment and allowed owner names. The tool receives the complete dependency object through `RunContext`. This is dependency injection, not memory.

In [ ]:
@dataclass
class IncidentDeps:
    environment: str
    runs: dict[str, dict]
    aliases: dict[str, str]
    allowed_owners: set[str]
    tool_audit: list[dict] = field(default_factory=list)
    validation_audit: list[dict] = field(default_factory=list)


def make_incident_deps() -> IncidentDeps:
    return IncidentDeps(
        environment="production",
        runs={
            "run-204": {
                "run_id": "run-204",
                "pipeline": "nightly-orders",
                "status": "failed",
                "failed_step": "merge_orders",
                "rows_lost": 1842,
                "owner": "finance-data",
                "last_checkpoint": "2026-08-19T23:45:00Z",
            }
        },
        aliases={"nightly-orders": "run-204"},
        allowed_owners={"data-platform", "finance-data", "security"},
    )

incident_deps = make_incident_deps()
print(incident_deps.runs["run-204"])

### The output type is a contract, not a policy

Pydantic will reject missing fields, invalid owner names, and unknown severity values. It cannot know that losing rows requires human review or that the owner must match the operational record. Those are business invariants for the output validator.

In [ ]:
class IncidentDecision(BaseModel):
    run_id: str
    severity: Literal["low", "medium", "high", "critical"]
    owner: Literal["data-platform", "finance-data", "security"]
    action: str
    requires_human: bool

print(IncidentDecision.model_json_schema())

### One reusable typed agent

In type-checking terms, this agent is `Agent[IncidentDeps, IncidentDecision]`. At runtime, PydanticAI also uses those types to construct schemas and validate values.

The instructions require an evidence tool call and explain the repair protocol. They do not contain the incident record itself.

In [ ]:
incident_agent = Agent(
    openai_model,
    name="pipeline_incident_agent",
    deps_type=IncidentDeps,
    output_type=IncidentDecision,
    instructions=(
        "Investigate one pipeline incident. First call lookup_pipeline_run using "
        "the identifier exactly as the user wrote. If the tool gives a canonical ID, "
        "call it again with that ID. Base the typed decision only on tool evidence. "
        "Data loss must be high or critical severity and requires human review."
    ),
    model_settings=openai_settings,
    retries=3,
)

@incident_agent.instructions
def deployment_context(ctx: RunContext[IncidentDeps]) -> str:
    owners = ", ".join(sorted(ctx.deps.allowed_owners))
    return f"Environment: {ctx.deps.environment}. Allowed owners: {owners}."

print(incident_agent.name)

### A tool can request a model repair

The operational lookup requires canonical run IDs. Humans use the pipeline alias `nightly-orders`, so the first call is expected to fail.

Raising `ModelRetry` does not hide the failure. PydanticAI adds a `RetryPromptPart` tied to that tool call and asks the model to correct its argument. The application still controls whether the tool returns data.

In [ ]:
@incident_agent.tool(retries=2)
def lookup_pipeline_run(ctx: RunContext[IncidentDeps], run_id: str) -> dict:
    """Return one pipeline run by its canonical run ID."""
    ctx.deps.tool_audit.append({"attempted_run_id": run_id})

    if run_id in ctx.deps.aliases:
        canonical = ctx.deps.aliases[run_id]
        raise ModelRetry(
            f"Use canonical run ID {canonical} and call this tool again."
        )
    if run_id not in ctx.deps.runs:
        raise ModelRetry(
            f"Unknown run ID. Valid IDs: {sorted(ctx.deps.runs)}"
        )
    return ctx.deps.runs[run_id]

### Validate meaning after validating shape

The output validator receives an already parsed `IncidentDecision`. It compares that object with trusted application data. If a policy invariant fails, `ModelRetry` sends a correction back to the model.

This is still bounded model self-correction, not enforcement by prompt. The validator is the enforcement code.

In [ ]:
@incident_agent.output_validator
def validate_incident_decision(
    ctx: RunContext[IncidentDeps],
    decision: IncidentDecision,
) -> IncidentDecision:
    ctx.deps.validation_audit.append(decision.model_dump())
    record = ctx.deps.runs.get(decision.run_id)
    problems = []

    if record is None:
        problems.append("run_id must be a canonical ID returned by the tool")
    else:
        if decision.owner != record["owner"]:
            problems.append(f"owner must be {record['owner']}")
        if record["rows_lost"] > 0 and decision.severity not in {"high", "critical"}:
            problems.append("data loss requires high or critical severity")
        if record["rows_lost"] > 0 and not decision.requires_human:
            problems.append("data loss requires human review")

    if problems:
        raise ModelRetry("; ".join(problems))
    return decision

### Bound the run and capture its protocol

Limits are part of the call site because different workflows can afford different budgets. This run permits enough requests for the planned alias repair and a possible output repair, but it cannot loop indefinitely.

`capture_run_messages()` records model protocol objects locally. It does not send data to Pydantic Logfire or any other hosted observability service.

In [ ]:
run_limits = UsageLimits(
    request_limit=6,
    tool_calls_limit=2,
    total_tokens_limit=4000,
)

with capture_run_messages() as captured_messages:
    incident_result = await incident_agent.run(
        "Investigate nightly-orders and recommend the next action.",
        deps=incident_deps,
        usage_limits=run_limits,
    )

print(incident_result.output)

## 3. Observe — inspect repair, output, usage, and tests

The final value is a Pydantic object. An IDE and static type checker know its fields, and runtime validation has already run. That is stronger than receiving an arbitrary JSON string, but only because the deterministic validator also checked business meaning.

In [ ]:
decision = incident_result.output
print("type:", type(decision).__name__)
print("validated dict:", decision.model_dump())
print("tool attempts:", incident_deps.tool_audit)
print("validator attempts:", len(incident_deps.validation_audit))

assert decision.run_id == "run-204"
assert decision.owner == "finance-data"
assert decision.severity in {"high", "critical"}
assert decision.requires_human is True

### Read the protocol as typed parts

The repair is visible between two model responses. The final structured output is also represented as a tool-shaped `final_result` payload on Chat Completions, then parsed into `IncidentDecision`. The output tool is a transport mechanism; it is not an application side effect.

In [ ]:
all_messages = incident_result.all_messages()
assert list(captured_messages) == all_messages

for index, message in enumerate(all_messages, start=1):
    print(index, type(message).__name__, [type(part).__name__ for part in message.parts])
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            print("  model tool call:", part.tool_name, part.args)
        elif isinstance(part, RetryPromptPart):
            print("  application retry:", part.tool_name, part.content)
        elif isinstance(part, ToolReturnPart):
            print("  application return:", part.tool_name, part.content)

### Usage distinguishes attempts from successful calls

The local tool audit contains two attempts: alias, then canonical ID. PydanticAI's `tool_calls` usage counts the one successful tool invocation; the retry attempt raised before returning data. Requests and tokens remain cumulative across the whole run.

PydanticAI also computes cost through its model-price data. We compare it with the course's pinned price variables so students can see where cost authority lives.

In [ ]:
usage = incident_result.usage
course_estimate = (
    usage.input_tokens * PRICE_INPUT
    + usage.output_tokens * PRICE_OUTPUT
) / 1_000_000

print(usage)
print("framework cost:", usage.cost)
print(f"course price estimate: ${course_estimate:.8f}")
print("tool attempts in application audit:", len(incident_deps.tool_audit))
print("successful tool calls in framework usage:", usage.tool_calls)

### Test application behavior without a model request

A useful framework must let ordinary tests replace nondeterministic infrastructure. `FunctionModel` accepts a Python function that returns model-protocol objects. `Agent.override` temporarily swaps the production model, while `ALLOW_MODEL_REQUESTS=False` prevents an accidental live call.

This test does not measure model quality. It proves the tool, dependency injection, typed output, validator, and application assertions work without network access or API cost.

In [ ]:
def deterministic_incident_model(messages, info: AgentInfo) -> ModelResponse:
    successful_lookup = any(
        isinstance(part, ToolReturnPart) and part.tool_name == "lookup_pipeline_run"
        for message in messages
        for part in message.parts
    )

    if not successful_lookup:
        return ModelResponse(parts=[
            ToolCallPart("lookup_pipeline_run", {"run_id": "run-204"})
        ])

    output_tool_name = info.output_tools[0].name
    return ModelResponse(parts=[
        ToolCallPart(output_tool_name, {
            "run_id": "run-204",
            "severity": "high",
            "owner": "finance-data",
            "action": "Restore from the checkpoint and reconcile row counts.",
            "requires_human": True,
        })
    ])

In [ ]:
test_deps = make_incident_deps()
previous_request_setting = models.ALLOW_MODEL_REQUESTS
models.ALLOW_MODEL_REQUESTS = False
try:
    with incident_agent.override(model=FunctionModel(deterministic_incident_model)):
        deterministic_result = await incident_agent.run(
            "Investigate run-204.",
            deps=test_deps,
        )
finally:
    models.ALLOW_MODEL_REQUESTS = previous_request_setting

assert deterministic_result.output.owner == "finance-data"
assert deterministic_result.output.requires_human is True
assert test_deps.tool_audit == [{"attempted_run_id": "run-204"}]
assert deterministic_result.usage.cost is None
print("Deterministic test passed with no provider request.")
print(deterministic_result.usage)

### What PydanticAI bought—and what it did not

| Concern | Framework contribution | Still application or platform work |
|---|---|---|
| Dependencies | typed injection through `RunContext` | lifecycle, secrets, authorization |
| Tool calls | schema, argument validation, retry protocol | allow/deny decisions and side-effect safety |
| Final output | parsing and Pydantic validation | factual checks and business invariants |
| Run limits | per-run request, token, tool, and cost bounds | organizational quotas and billing alerts |
| Observability | typed messages and optional integrations | governed retention and access control |
| Testing | model override and deterministic model doubles | model-quality evals and production monitoring |

This is a framework for typed Python applications, not a replacement for architecture.

## 4. Challenge — add a typed recovery decision

Build a second PydanticAI agent for this record:

```python
{
    "run_id": "run-305",
    "pipeline": "customer-dim-refresh",
    "status": "degraded",
    "rows_delayed": 25000,
    "owner": "data-platform",
    "sla_minutes_remaining": 45,
}
```

Acceptance criteria:

1. Define a typed `RecoveryDecision` with `run_id`, `priority` (`p1`, `p2`, or `p3`), `owner`, `action`, and `requires_human`.
2. Pass the record through typed dependencies; do not paste it into the prompt.
3. Register `lookup_recovery_run` and require the model to call it.
4. Add an output validator: the owner must match trusted data; more than 10,000 delayed rows must be `p1` or `p2` and require a person.
5. Run with explicit request, tool-call, and total-token limits.
6. Store the result in `challenge_result` and the dependencies in `challenge_deps`.

The separate solution notebook contains one answer.

In [ ]:
# Build your RecoveryDecision, dependencies, tool, validator, and run here.
# The verification cell expects:
#   challenge_result
#   challenge_deps

# class RecoveryDecision(BaseModel):
#     ...

# challenge_agent = Agent(...)
# ...

In [ ]:
challenge_output = challenge_result.output
challenge_lookup_calls = [
    part
    for message in challenge_result.all_messages()
    for part in message.parts
    if isinstance(part, ToolCallPart) and part.tool_name == "lookup_recovery_run"
]

assert isinstance(challenge_output, RecoveryDecision)
assert challenge_output.run_id == "run-305"
assert challenge_output.owner == "data-platform"
assert challenge_output.priority in {"p1", "p2"}
assert challenge_output.requires_human is True
assert len(challenge_lookup_calls) == 1
assert challenge_deps.tool_audit == [{"attempted_run_id": "run-305"}]
print("Challenge passed: delayed pipeline received a typed, grounded recovery decision.")

## Takeaway

PydanticAI's value is not that a Pydantic model makes an LLM correct. Its value is that dependencies, tool arguments, outputs, repair paths, limits, messages, and test doubles become explicit Python application surfaces.

Students should leave able to explain four separate checks:

1. type validation asks whether a value has the required shape;
2. a tool asks trusted systems for evidence;
3. an output validator enforces business meaning;
4. deterministic tests prove application behavior without pretending to evaluate the production model.